# PP Model-PP

Read the root `README.md` first. / . 

The same six crops, weather and interface are used in both modes. / Mode, . 
Storage-organ dry matter is reported. No nitrogen leaching is calculated. / , . 


In [ ]:
from pathlib import Path
import runpy
import sys
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '06_Hao_Interface_and_Rerun').is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the full handoff folder. /  Jupyter. ")
registry = runpy.run_path(str(ROOT / '06_Hao_Interface_and_Rerun/Shared_Core/Module_Loader.py'))
registry["install"]()
from crop_engine import ModelConfig, GroundGrid, crop_catalog, prepare_weather, evaluate
registry["load_module"]("radiation_bridge")
from radiation_bridge import preflight, get_defaults, run_tianze_daily
print("Project folder / :", ROOT)
print("Radiance status / :", preflight())


## 1. Crop and assumptions-
,  `CROP_IDS`  ID. . 

In [ ]:
MODE = "PP"
ACCEPT_DEMO_ASSUMPTIONS = True
CROP_IDS = ["spring_barley"]  # relocation execution check
GRID = GroundGrid()
display(pd.DataFrame(crop_catalog()).T[["label", "variety", "start_mm_dd", "end_mm_dd", "organ", "reference_open_field_yield_t_ha"]])


## 2. Run natural-light baseline-
; Winter wheat, .  TMY . 

In [ ]:
natural_results = {}
natural_folders = {}
if ACCEPT_DEMO_ASSUMPTIONS:
    for crop_id in CROP_IDS:
        config = ModelConfig(crop_id=crop_id, mode=MODE, accept_demo_assumptions=True)
        result, folder = evaluate(config, grid=GRID)
        natural_results[crop_id] = result
        natural_folders[crop_id] = folder
    display(pd.DataFrame([{k:r[k] for k in ["crop_id","mode","yield_kg_DM_ha","status","transpiration_mm","drainage_bottom_mm"]} for r in natural_results.values()]))
else:
    print(". ,  ACCEPT_DEMO_ASSUMPTIONS  True. ")


## 3. Inspect model results-
WLP ; PP . 

In [ ]:
if natural_folders:
    crop_id = CROP_IDS[0]
    folder = natural_folders[crop_id]
    display(pd.read_csv(folder / "Point_Summary.csv"))
    if (folder / "Daily_Soil_Layers.csv").exists():
        display(pd.read_csv(folder / "Daily_Soil_Layers.csv").head(20))
    else:
        display(pd.read_csv(folder / "Daily_Crop_Growth.csv").head())
    print(": ", folder)


## 4. Connect actual radiation-
.  `GRID`  Tianze . ,  LER . 

In [ ]:
RADIATION_CSV = None  # × CSV 
RADIATION_PROVENANCE = None  #  run_id
SCENARIO_ID = "fixed_conventional"
SELECTED_CROP = CROP_IDS[0]
if RADIATION_CSV is not None:
    if not ACCEPT_DEMO_ASSUMPTIONS or not RADIATION_PROVENANCE:
        raise ValueError(". ")
    config = ModelConfig(crop_id=SELECTED_CROP, mode=MODE, accept_demo_assumptions=True)
    result, folder = evaluate(config, radiation=RADIATION_CSV, scenario_id=SCENARIO_ID,
        provenance=RADIATION_PROVENANCE, grid=GRID, baseline=natural_results.get(SELECTED_CROP))
    display(pd.DataFrame([result]).drop(columns=["reference_table_row"], errors="ignore"))
    print(folder)
else:
    print(" APV ,  APV . ")


## 5. Optional Tianze radiation-Tianze
, . , . 

In [ ]:
RUN_SINGLE_DAY_RADIANCE = False
if RUN_SINGLE_DAY_RADIANCE:
    config = ModelConfig(crop_id=SELECTED_CROP, mode=MODE, accept_demo_assumptions=True)
    weather, weather_info = prepare_weather(config)
    defaults = get_defaults()["fixed"]
    radiation, source = run_tianze_daily("fixed", [weather.day.iloc[0]], GRID,
        design=defaults["design"], module_spec=defaults["module"], array_spec=defaults["array"],
        albedo=0.20, epw_path=ROOT / '03_Precipitation' / "Aachen_TMY_Weather.epw", allow_run=True,
        accuracy="high")
    display(radiation.head())
    print(source)
